
# Stage D (Part 1) — Build Static Basin Attributes

**Purpose**
Create a clean, modeling-ready table of **static geospatial attributes** per basin — including shape, topography, land cover, and soil composition — and save it for reuse in later stages.

---

## What this notebook does

1. **Load processed sources:**

   * `data/boundaries/processed/basin_shape_stats.csv`
   * `data/boundaries/processed/basin_dem_acc_stats.csv`
   * `data/boundaries/processed/bhutan_lulc_by_basin_km2_ORDERED_CLEAN.csv`
   * `data/boundaries/processed/bhutan_soil_wrb_by_basin_km2_ORDERED_CLEAN.csv`
2. Normalize basin names and merge sources; attach `basin_id` via
   `data/boundaries/processed/basin_lookup.csv`.
3. Compute **derived shape/topography features** (compactness, elevation range, coefficients of variation, log-scaled accumulation, slope-percent columns).
4. Compute **LULC composition**:

   * Convert land-cover areas (km²) into per-basin percentages (e.g. `trees_pct`, `crops_pct`, `builtarea_pct`).
   * Optionally keep raw km² columns.
5. Compute **Soil composition**:

   * Fill missing areas with zero.
   * Convert soil class areas into percentages (e.g. `cambisols_pct`, `gleysols_pct`).
6. Perform duplicate-key checks, assemble into one wide table, and sort by `basin_id`/`basin_name`.
7. Write the final static table and a provenance JSON for reproducibility.

---

## Inputs

* `data/boundaries/processed/basin_shape_stats.csv`
  *(centroid, area, perimeter)*
* `data/boundaries/processed/basin_dem_acc_stats.csv`
  *(DEM stats, slope, relief, accumulation)*
* `data/boundaries/processed/bhutan_lulc_by_basin_km2_ORDERED_CLEAN.csv`
  *(land-use / land-cover class areas per basin)*
* `data/boundaries/processed/bhutan_soil_wrb_by_basin_km2_ORDERED_CLEAN.csv`
  *(WRB soil class areas per basin)*
* `data/boundaries/processed/basin_lookup.csv`
  *(maps `basin_name ↔ basin_id`)*

---

## Outputs

* **Static attributes (all basins):**
  `data/modeling/static/basin_attributes.parquet`
* **Provenance:**
  `data/modeling/static/basin_attributes.meta.json`

---

## Notes

* **Shape metrics:**
  `compactness = 4π · area_sqkm / perimeter_km²` (0–1; higher = more compact)
  `perim_area_ratio = perimeter_km / √area_sqkm` (elongation proxy)

* **Topography metrics:**
  `elev_range_m = dem_max − dem_min`
  `dem_cv = dem_std / dem_mean` (guarded for zero mean)
  Accumulation fields stored as `log1p(*)` to stabilize scale.
  Two slope-fraction columns renamed to `pct_slope_gt_a` / `pct_slope_gt_b`.

* **LULC / Soil composition:**
  All `_km2` columns converted to `_pct` (share of total basin area).
  Missing values filled with 0 before computing percentages.

---

## How this is used next

These static attributes are **left-joined** onto each `(basin_id, date_local)` row when building **daily dynamic features** (Stage D Part 2) and then carried into the modeling datasets.


In [1]:
from pathlib import Path
import subprocess
import pandas as pd, numpy as np, json, re
from datetime import datetime

## Resolve Project Root

In [2]:
# === Resolve Project Root ===

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/qingfangliu/bhutan_climate_modeling


In [3]:
# === Paths & settings ===
SHAPE_CSV         = PROJECT_ROOT / "data/boundaries/processed/basin_shape_stats.csv"
DEM_ACC_CSV       = PROJECT_ROOT / "data/boundaries/processed/basin_dem_acc_stats.csv"
LOOKUP_CSV        = PROJECT_ROOT / "data/boundaries/processed/basin_lookup.csv"

SOIL_CSV          = PROJECT_ROOT / "data/boundaries/processed/bhutan_soil_wrb_by_basin_km2_ORDERED_CLEAN.csv"
LULC_CSV          = PROJECT_ROOT / "data/boundaries/processed/bhutan_lulc_by_basin_km2_ORDERED_CLEAN.csv"

In [4]:
OUT_DIR           = PROJECT_ROOT / "data/modeling/static"
OUT_PARQUET       = OUT_DIR / "basin_attributes.parquet"
OUT_META          = OUT_DIR / "basin_attributes.meta.json"

In [5]:
# Optional plug-in folder for future geospatial features you may add later
EXTRA_DIR         = OUT_DIR / "extra"     # drop CSV/Parquet here to auto-merge
ALLOW_OVERWRITE   = False                 # if an extra file has a column that already exists, append "_extra"

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# === Helpers ===
def basin_key_series(s: pd.Series) -> pd.Series:
    """Case/spacing-insensitive basin name key (handles underscores/spaces)."""
    return (s.astype(str)
             .str.strip()
             .str.replace(r"[_\s]+", " ", regex=True)
             .str.lower())

def first_existing(d: pd.DataFrame, names: list[str]) -> str:
    for n in names:
        if n in d.columns:
            return n
    raise KeyError(f"None of the expected columns {names} found in: {list(d.columns)}")

def compactness(area_sqkm: pd.Series, perimeter_km: pd.Series) -> pd.Series:
    # 4πA/P², unitless (consistent if A in km² and P in km)
    return (4*np.pi*area_sqkm) / np.where(perimeter_km>0, perimeter_km**2, np.nan)

def perim_area_ratio(perimeter_km: pd.Series, area_sqkm: pd.Series) -> pd.Series:
    return perimeter_km / np.sqrt(np.where(area_sqkm>0, area_sqkm, np.nan))


## Load & clean basin_shape_stats.csv

In [7]:
# === Load shape stats ===
shape = pd.read_csv(SHAPE_CSV)

# Flexible column discovery (your header preview showed "perimeter_km" or "perimeter_kr")
lon_col = first_existing(shape, ["longitude","lon","LONGITUDE"])
lat_col = first_existing(shape, ["latitude","lat","LATITUDE"])
area_col = first_existing(shape, ["area_sqkm","area_km2","AREA_SQKM"])
perim_col = first_existing(shape, ["perimeter_km","perimeter_kr","perimeter_kilometers","PERIMETER_KM"])
basin_name_col = first_existing(shape, ["basin","Basin","basin_name","BASIN_NAME"])

shape = shape.rename(columns={
    basin_name_col: "basin_name",
    lon_col: "longitude",
    lat_col: "latitude",
    area_col: "area_sqkm",
    perim_col: "perimeter_km",
})

# Keep only what we need; compute shape-derived features
shape = shape[["basin_name","longitude","latitude","area_sqkm","perimeter_km"]].copy()
shape["compactness"]      = compactness(shape["area_sqkm"], shape["perimeter_km"])
shape["perim_area_ratio"] = perim_area_ratio(shape["perimeter_km"], shape["area_sqkm"])

shape["basin_key"] = basin_key_series(shape["basin_name"])

# Check duplicates by key
dups = shape["basin_key"].value_counts()
if (dups>1).any():
    raise ValueError(f"Duplicate basins in shape stats for keys:\n{dups[dups>1]}")


In [8]:
shape

,basin_name,longitude,latitude,area_sqkm,perimeter_km,compactness,perim_area_ratio,basin_key
0,Aiechhu,90.464808,26.945874,1963.905886,0.002897,2.940004e+09,0.000065,aiechhu
1,Merak_Sakteng,92.033958,27.329946,138.810677,0.000645,4.188638e+09,0.000055,merak sakteng
2,Mangdechhu,90.649025,27.506149,7436.605311,0.004425,4.771824e+09,0.000051,mangdechhu
3,Jaldhakha,88.987850,27.061616,1038.942931,0.002133,2.869328e+09,0.000066,jaldhakha
4,Amochhu,89.120110,27.350269,3929.619367,0.004123,2.904399e+09,0.000066,amochhu
5,Wangchhu,89.482723,27.353115,4608.336971,0.004038,3.551970e+09,0.000059,wangchhu
6,Drangmechhu,91.395029,27.842435,21098.266822,0.009023,3.256464e+09,0.000062,drangmechhu
7,Punatsangchhu,89.939466,27.562875,9765.747380,0.005941,3.476406e+09,0.000060,punatsangchhu
8,Nyera_Amari,91.652809,26.987295,2262.233550,0.003570,2.230784e+09,0.000075,nyera amari
9,Jomori,91.959758,27.123484,750.319391,0.001549,3.931682e+09,0.000057,jomori


## Load & clean basin_dem_acc_stats.csv

In [9]:
# === Load DEM + accumulation stats ===
dem = pd.read_csv(DEM_ACC_CSV)

basin_name_col = first_existing(dem, ["basin","Basin","basin_name","BASIN_NAME"])
dem = dem.rename(columns={basin_name_col: "basin_name"})

# Flexible column picks
cols_expected = {
    "dem_min":      first_existing(dem, ["dem_min","DEM_MIN"]),
    "dem_max":      first_existing(dem, ["dem_max","DEM_MAX"]),
    "dem_mean":     first_existing(dem, ["dem_mean","DEM_MEAN"]),
    "dem_std":      first_existing(dem, ["dem_std","DEM_STD"]),
    "dem_median":   first_existing(dem, ["dem_median","DEM_MEDIAN"]),
    "relief_m":     first_existing(dem, ["relief_m","RELIEF_M"]),
    "slope_mean":   first_existing(dem, ["slope_mean","slope_mean_deg"]),
    "slope_p90_deg":first_existing(dem, ["slope_p90_deg","slope_p90","SLOPE_P90_DEG","SLOPE_P90"]),
}
# Optional accumulation & slope-fraction columns (best-judgment)
acc_mean_col = next((c for c in ["acc_mean","ACC_MEAN"] if c in dem.columns), None)
acc_max_col  = next((c for c in ["acc_max","ACC_MAX"]   if c in dem.columns), None)

# Two unknown slope % columns (keep if present)
pct_cols = [c for c in dem.columns if re.match(r"^pct_slope", c, flags=re.IGNORECASE)]
pct_cols = pct_cols[:2]  # keep at most two

# Build tight frame
keep = ["basin_name"] + list(cols_expected.values()) + [c for c in [acc_mean_col, acc_max_col] if c] + pct_cols
dem = dem[keep].copy()
dem = dem.rename(columns={v:k for k,v in cols_expected.items()})

if acc_mean_col: dem = dem.rename(columns={acc_mean_col:"acc_mean"})
if acc_max_col:  dem = dem.rename(columns={acc_max_col:"acc_max"})

# Derived
dem["elev_range_m"] = dem["dem_max"] - dem["dem_min"]
dem["dem_cv"]       = dem["dem_std"] / dem["dem_mean"].replace({0:np.nan})

# Stabilized logs for accumulation (only if available)
if "acc_mean" in dem.columns:
    dem["log_acc_mean"] = np.log1p(dem["acc_mean"].clip(lower=0))
if "acc_max" in dem.columns:
    dem["log_acc_max"]  = np.log1p(dem["acc_max"].clip(lower=0))

# Rename the two slope-fraction columns (if present) to generic names
if len(pct_cols) >= 1: dem = dem.rename(columns={pct_cols[0]: "pct_slope_gt_a"})
if len(pct_cols) >= 2: dem = dem.rename(columns={pct_cols[1]: "pct_slope_gt_b"})

dem["basin_key"] = basin_key_series(dem["basin_name"])

# Check duplicates
dups = dem["basin_key"].value_counts()
if (dups>1).any():
    raise ValueError(f"Duplicate basins in DEM/acc stats for keys:\n{dups[dups>1]}")


In [10]:
dem

,basin_name,dem_min,dem_max,dem_mean,dem_std,dem_median,relief_m,slope_mean,slope_p90_deg,acc_mean,acc_max,pct_slope_gt_a,pct_slope_gt_b,elev_range_m,dem_cv,log_acc_mean,log_acc_max,basin_key
0,Aiechhu,92.0,4160.0,1183.538707,699.525166,1091.0,4068.0,21.200437,34.513462,210.078824,3754703.0,72.055599,21.698405,4068.0,0.591045,5.352232,15.138520,aiechhu
1,Merak_Sakteng,2690.0,4483.0,3882.785851,340.278285,3965.0,1793.0,20.382105,31.582006,65.099067,10252.0,71.324777,13.740984,1793.0,0.087638,4.191155,9.235326,merak sakteng
2,Mangdechhu,109.0,7065.0,3230.588248,1302.878378,3287.0,6956.0,23.063430,35.971821,1070.502172,979449.0,77.342031,25.226937,6956.0,0.403294,6.976817,13.794746,mangdechhu
3,Jaldhakha,210.0,4577.0,1588.797296,1011.341252,1403.0,4367.0,21.281973,33.426529,199.115628,72476.0,74.670477,18.176001,4367.0,0.636545,5.298895,11.191025,jaldhakha
4,Amochhu,164.0,6689.0,3187.160344,1411.272228,3521.0,6525.0,22.034436,34.454346,894.656866,497461.0,75.496564,21.330364,6525.0,0.442799,6.797557,13.117274,amochhu
5,Wangchhu,117.0,6689.0,3228.431849,1081.635175,3270.0,6572.0,23.030790,34.844448,1076.646375,602706.0,80.520147,22.038662,6572.0,0.335034,6.982535,13.309186,wangchhu
6,Drangmechhu,90.0,7127.0,3824.023227,1417.195291,4292.0,7037.0,22.540061,35.928204,1944.081318,3756205.0,74.820652,24.171486,7037.0,0.370603,7.573059,15.138920,drangmechhu
7,Punatsangchhu,94.0,7087.0,3149.052890,1405.579514,3117.0,6993.0,23.648238,36.667324,1444.768641,1277715.0,79.314210,26.663519,6993.0,0.446350,7.276396,14.060585,punatsangchhu
8,Nyera_Amari,99.0,4462.0,1536.849713,1003.422671,1391.0,4363.0,22.749070,36.319275,313.954688,145267.0,75.249209,25.221410,4363.0,0.652909,5.752429,11.886336,nyera amari
9,Jomori,217.0,4477.0,2239.513228,918.928038,2243.0,4260.0,26.134153,37.785282,408.963529,95581.0,88.162490,33.790817,4260.0,0.410325,6.016068,11.467740,jomori


## Read and Prepare LULC Data

In [11]:
# === Load LULC ===
lulc = pd.read_csv(LULC_CSV)

# Standardize column names: basin name + area classes
lulc = lulc.rename(columns={"basin": "basin_name"})

# Calculate percentages for each land cover class
lulc_classes = [c for c in lulc.columns if c.endswith("_km2") and c != "total_km2"]

for c in lulc_classes:
    lulc[f"{c.replace('_km2','')}_pct"] = lulc[c] / lulc["total_km2"]

# Add basin_key for joining
lulc["basin_key"] = basin_key_series(lulc["basin_name"])

# Quick duplicate check
dups = lulc["basin_key"].value_counts()
if (dups > 1).any():
    raise ValueError(f"Duplicate basins in LULC for keys:\n{dups[dups>1]}")


In [12]:
lulc

,basin_name,Water_km2,Trees_km2,FloodedVegetation_km2,Crops_km2,BuiltArea_km2,BareGround_km2,SnowIce_km2,Rangeland_km2,total_km2,Water_pct,Trees_pct,FloodedVegetation_pct,Crops_pct,BuiltArea_pct,BareGround_pct,SnowIce_pct,Rangeland_pct,basin_key
0,Aiechhu,2.731,1823.859,0.000,9.784,29.243,9.347,0.000,52.909,1927.873,0.001417,0.946047,0.000000e+00,0.005075,0.015169,0.004848,0.000000,0.027444,aiechhu
1,Merak_Sakte,0.342,72.706,0.000,0.000,0.001,0.076,0.000,60.442,133.567,0.002561,0.544341,0.000000e+00,0.000000,0.000007,0.000569,0.000000,0.452522,merak sakte
2,Mangdechhu,44.139,5135.477,0.025,13.669,18.519,349.743,414.462,1462.231,7438.265,0.005934,0.690413,3.360999e-06,0.001838,0.002490,0.047019,0.055720,0.196582,mangdechhu
3,Jaldhakha,1.647,828.163,0.000,4.859,11.507,4.526,0.000,55.142,905.844,0.001818,0.914245,0.000000e+00,0.005364,0.012703,0.004996,0.000000,0.060874,jaldhakha
4,Amochhu,7.993,1929.189,0.000,1.265,20.779,21.870,1.207,333.941,2316.244,0.003451,0.832895,0.000000e+00,0.000546,0.008971,0.009442,0.000521,0.144173,amochhu
5,Wangchhu,11.094,2917.059,0.000,8.896,135.771,154.039,26.339,1354.175,4607.373,0.002408,0.633128,0.000000e+00,0.001931,0.029468,0.033433,0.005717,0.293915,wangchhu
6,Drangmechhu,37.982,6698.338,0.006,6.052,115.035,64.653,346.655,1224.898,8493.619,0.004472,0.788632,7.064127e-07,0.000713,0.013544,0.007612,0.040814,0.144214,drangmechhu
7,Punatsangchhu,58.939,6074.535,0.008,32.923,73.500,429.553,735.391,2289.215,9694.064,0.006080,0.626624,8.252473e-07,0.003396,0.007582,0.044311,0.075860,0.236146,punatsangchhu
8,Nyera_Amari,2.978,2023.385,0.000,6.961,21.663,3.432,0.000,128.788,2187.207,0.001362,0.925100,0.000000e+00,0.003183,0.009904,0.001569,0.000000,0.058882,nyera amari
9,Jomori,0.664,634.884,0.000,0.554,5.152,0.062,0.000,81.218,722.534,0.000919,0.878691,0.000000e+00,0.000767,0.007130,0.000086,0.000000,0.112407,jomori


## Read and Prepare Soil Data

In [13]:
# === Load Soil ===
soil = pd.read_csv(SOIL_CSV)

soil = soil.rename(columns={"basin": "basin_name"})

# Treat NaNs as 0 (missing soil class area)
soil = soil.fillna(0)

soil_classes = [c for c in soil.columns if c.endswith("_km2") and c != "total_km2"]

for c in soil_classes:
    soil[f"{c.replace('_km2','')}_pct"] = soil[c] / soil["total_km2"]

soil["basin_key"] = basin_key_series(soil["basin_name"])

# Duplicate check
dups = soil["basin_key"].value_counts()
if (dups > 1).any():
    raise ValueError(f"Duplicate basins in SOIL for keys:\n{dups[dups>1]}")


In [14]:
soil

,basin_name,Acrisols_km2,Albeluvisols_km2,Alisols_km2,Andosols_km2,Arenosols_km2,Calcisols_km2,Cambisols_km2,Chernozems_km2,Cryosols_km2,...,Planosols_pct,Plinthosols_pct,Podzols_pct,Regosols_pct,Solonchaks_pct,Solonetz_pct,Stagnosols_pct,Umbrisols_pct,Vertisols_pct,basin_key
0,Aiechhu,0.0,0.0,0.500,0.0,0.0,0.0,794.562,0.0,0.000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000763,aiechhu
1,Merak_Sakte,0.0,0.0,0.000,0.0,0.0,0.0,0.438,0.0,0.000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,merak sakte
2,Mangdechhu,0.0,0.0,0.125,0.0,0.0,0.0,1071.125,0.0,275.188,...,0.0,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.000000,mangdechhu
3,Jaldhakha,0.0,0.0,2.312,0.0,0.0,0.0,229.125,0.0,0.000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000208,jaldhakha
4,Amochhu,0.0,0.0,1.250,0.0,0.0,0.0,315.625,0.0,1.000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000106,amochhu
5,Wangchhu,0.0,0.0,0.688,0.0,0.0,0.0,259.750,0.0,17.125,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000014,wangchhu
6,Drangmechhu,0.0,0.0,0.000,0.0,0.0,0.0,1879.500,0.0,300.500,...,0.0,0.0,0.000022,0.0,0.0,0.0,0.0,0.0,0.000000,drangmechhu
7,Punatsangchhu,0.0,0.0,0.000,0.0,0.0,0.0,1245.188,0.0,369.938,...,0.0,0.0,0.000007,0.0,0.0,0.0,0.0,0.0,0.000043,punatsangchhu
8,Nyera_Amari,0.0,0.0,3.562,0.0,0.0,0.0,808.562,0.0,0.000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,nyera amari
9,Jomori,0.0,0.0,1.188,0.0,0.0,0.0,126.125,0.0,0.000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,jomori


## Merge shape + DEM stats → attach basin_id

In [15]:
# Keep 'basin_name' only from DEM as the canonical name
lulc_nodup = lulc.drop(columns=["basin_name"])
soil_nodup = soil.drop(columns=["basin_name"])
shape_nodup = shape.drop(columns=["basin_name"])

# Merge on basin_key
all_features = (
    dem
    .merge(lulc_nodup, on="basin_key", how="left")
    .merge(soil_nodup, on="basin_key", how="left")
    .merge(shape_nodup, on="basin_key", how="left")
)

all_features

,basin_name,dem_min,dem_max,dem_mean,dem_std,dem_median,relief_m,slope_mean,slope_p90_deg,acc_mean,...,Solonetz_pct,Stagnosols_pct,Umbrisols_pct,Vertisols_pct,longitude,latitude,area_sqkm,perimeter_km,compactness,perim_area_ratio
0,Aiechhu,92.0,4160.0,1183.538707,699.525166,1091.0,4068.0,21.200437,34.513462,210.078824,...,0.0,0.0,0.0,0.000763,90.464808,26.945874,1963.905886,0.002897,2.940004e+09,0.000065
1,Merak_Sakteng,2690.0,4483.0,3882.785851,340.278285,3965.0,1793.0,20.382105,31.582006,65.099067,...,NaN,NaN,NaN,NaN,92.033958,27.329946,138.810677,0.000645,4.188638e+09,0.000055
2,Mangdechhu,109.0,7065.0,3230.588248,1302.878378,3287.0,6956.0,23.063430,35.971821,1070.502172,...,0.0,0.0,0.0,0.000000,90.649025,27.506149,7436.605311,0.004425,4.771824e+09,0.000051
3,Jaldhakha,210.0,4577.0,1588.797296,1011.341252,1403.0,4367.0,21.281973,33.426529,199.115628,...,0.0,0.0,0.0,0.000208,88.987850,27.061616,1038.942931,0.002133,2.869328e+09,0.000066
4,Amochhu,164.0,6689.0,3187.160344,1411.272228,3521.0,6525.0,22.034436,34.454346,894.656866,...,0.0,0.0,0.0,0.000106,89.120110,27.350269,3929.619367,0.004123,2.904399e+09,0.000066
5,Wangchhu,117.0,6689.0,3228.431849,1081.635175,3270.0,6572.0,23.030790,34.844448,1076.646375,...,0.0,0.0,0.0,0.000014,89.482723,27.353115,4608.336971,0.004038,3.551970e+09,0.000059
6,Drangmechhu,90.0,7127.0,3824.023227,1417.195291,4292.0,7037.0,22.540061,35.928204,1944.081318,...,0.0,0.0,0.0,0.000000,91.395029,27.842435,21098.266822,0.009023,3.256464e+09,0.000062
7,Punatsangchhu,94.0,7087.0,3149.052890,1405.579514,3117.0,6993.0,23.648238,36.667324,1444.768641,...,0.0,0.0,0.0,0.000043,89.939466,27.562875,9765.747380,0.005941,3.476406e+09,0.000060
8,Nyera_Amari,99.0,4462.0,1536.849713,1003.422671,1391.0,4363.0,22.749070,36.319275,313.954688,...,0.0,0.0,0.0,0.000000,91.652809,26.987295,2262.233550,0.003570,2.230784e+09,0.000075
9,Jomori,217.0,4477.0,2239.513228,918.928038,2243.0,4260.0,26.134153,37.785282,408.963529,...,0.0,0.0,0.0,0.000000,91.959758,27.123484,750.319391,0.001549,3.931682e+09,0.000057


In [16]:
# Attach numeric basin_id (authoritative join key downstream)
lookup = pd.read_csv(LOOKUP_CSV)
name_col = first_existing(lookup, ["basin_name","BASIN_NAME","basin"])
lookup = lookup.rename(columns={name_col: "basin_name"})
lookup["basin_key"] = basin_key_series(lookup["basin_name"])
lookup

,basin_id,basin_name,basin_slug,basin_key
0,1.0,Aiechhu,aiechhu,aiechhu
1,3.0,Mangdechhu,mangdechhu,mangdechhu
2,4.0,Jaldhakha,jaldhakha,jaldhakha
3,5.0,Amochhu,amochhu,amochhu
4,6.0,Wangchhu,wangchhu,wangchhu
5,7.0,Drangmechhu,drangmechhu,drangmechhu
6,8.0,Punatsangchhu,punatsangchhu,punatsangchhu
7,9.0,Nyera_Amari,nyera_amari,nyera amari
8,10.0,Jomori,jomori,jomori


In [17]:
static = all_features.merge(lookup[["basin_key","basin_name","basin_id"]],
                       on="basin_key", how="left", suffixes=("", "_lkp"))

# Prefer lookup display name where available
static["basin_name"] = np.where(static["basin_name_lkp"].notna(), static["basin_name_lkp"], static["basin_name"])
static = static.drop(columns=["basin_name_lkp"])

# QA: report any missing basin_id
missing_id = static[static["basin_id"].isna()][["basin_name"]].sort_values("basin_name").drop_duplicates()
if len(missing_id):
    print("WARNING: These basins did not find a basin_id in lookup:")
    display(missing_id.head(20))

,basin_name
1,Merak_Sakteng


In [18]:
static

,basin_name,dem_min,dem_max,dem_mean,dem_std,dem_median,relief_m,slope_mean,slope_p90_deg,acc_mean,...,Stagnosols_pct,Umbrisols_pct,Vertisols_pct,longitude,latitude,area_sqkm,perimeter_km,compactness,perim_area_ratio,basin_id
0,Aiechhu,92.0,4160.0,1183.538707,699.525166,1091.0,4068.0,21.200437,34.513462,210.078824,...,0.0,0.0,0.000763,90.464808,26.945874,1963.905886,0.002897,2.940004e+09,0.000065,1.0
1,Merak_Sakteng,2690.0,4483.0,3882.785851,340.278285,3965.0,1793.0,20.382105,31.582006,65.099067,...,NaN,NaN,NaN,92.033958,27.329946,138.810677,0.000645,4.188638e+09,0.000055,NaN
2,Mangdechhu,109.0,7065.0,3230.588248,1302.878378,3287.0,6956.0,23.063430,35.971821,1070.502172,...,0.0,0.0,0.000000,90.649025,27.506149,7436.605311,0.004425,4.771824e+09,0.000051,3.0
3,Jaldhakha,210.0,4577.0,1588.797296,1011.341252,1403.0,4367.0,21.281973,33.426529,199.115628,...,0.0,0.0,0.000208,88.987850,27.061616,1038.942931,0.002133,2.869328e+09,0.000066,4.0
4,Amochhu,164.0,6689.0,3187.160344,1411.272228,3521.0,6525.0,22.034436,34.454346,894.656866,...,0.0,0.0,0.000106,89.120110,27.350269,3929.619367,0.004123,2.904399e+09,0.000066,5.0
5,Wangchhu,117.0,6689.0,3228.431849,1081.635175,3270.0,6572.0,23.030790,34.844448,1076.646375,...,0.0,0.0,0.000014,89.482723,27.353115,4608.336971,0.004038,3.551970e+09,0.000059,6.0
6,Drangmechhu,90.0,7127.0,3824.023227,1417.195291,4292.0,7037.0,22.540061,35.928204,1944.081318,...,0.0,0.0,0.000000,91.395029,27.842435,21098.266822,0.009023,3.256464e+09,0.000062,7.0
7,Punatsangchhu,94.0,7087.0,3149.052890,1405.579514,3117.0,6993.0,23.648238,36.667324,1444.768641,...,0.0,0.0,0.000043,89.939466,27.562875,9765.747380,0.005941,3.476406e+09,0.000060,8.0
8,Nyera_Amari,99.0,4462.0,1536.849713,1003.422671,1391.0,4363.0,22.749070,36.319275,313.954688,...,0.0,0.0,0.000000,91.652809,26.987295,2262.233550,0.003570,2.230784e+09,0.000075,9.0
9,Jomori,217.0,4477.0,2239.513228,918.928038,2243.0,4260.0,26.134153,37.785282,408.963529,...,0.0,0.0,0.000000,91.959758,27.123484,750.319391,0.001549,3.931682e+09,0.000057,10.0


## Reorder/limit columns, write outputs, and a quick preview

In [19]:
# === Column order & write ===
base_cols = [
    "basin_id","basin_name",
    "longitude","latitude",
    "area_sqkm","perimeter_km","compactness","perim_area_ratio",
    "dem_min","dem_max","dem_mean","dem_median","dem_std","elev_range_m","dem_cv",
    "slope_mean","slope_p90_deg","relief_m",
    "log_acc_mean","log_acc_max",
    "pct_slope_gt_a","pct_slope_gt_b",
]

# Keep existing ones in order, then append any extras at the end
ordered = [c for c in base_cols if c in static.columns]
extra   = [c for c in static.columns if c not in ordered + ["basin_key"]]
final_df = static[ordered + extra].copy()

In [20]:
# Persist
final_df = final_df.sort_values(["basin_id","basin_name"], na_position="last").reset_index(drop=True)
final_df.to_parquet(OUT_PARQUET, index=False)

meta = {
    "created_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "source_files": [str(SHAPE_CSV.relative_to(PROJECT_ROOT)),
                     str(DEM_ACC_CSV.relative_to(PROJECT_ROOT)),
                     str(SOIL_CSV.relative_to(PROJECT_ROOT)),
                     str(LULC_CSV.relative_to(PROJECT_ROOT)),
                     str(LOOKUP_CSV.relative_to(PROJECT_ROOT))],
    "n_basins": int(final_df["basin_id"].notna().sum()),
    "columns": list(final_df.columns),
}
with open(OUT_META, "w") as f:
    json.dump(meta, f, indent=2)

print("Wrote:", OUT_PARQUET)
print("Wrote:", OUT_META)
display(final_df.head(10))

Wrote: /Users/qingfangliu/bhutan_climate_modeling/data/modeling/static/basin_attributes.parquet
Wrote: /Users/qingfangliu/bhutan_climate_modeling/data/modeling/static/basin_attributes.meta.json


,basin_id,basin_name,longitude,latitude,area_sqkm,perimeter_km,compactness,perim_area_ratio,dem_min,dem_max,...,Phaeozems_pct,Planosols_pct,Plinthosols_pct,Podzols_pct,Regosols_pct,Solonchaks_pct,Solonetz_pct,Stagnosols_pct,Umbrisols_pct,Vertisols_pct
0,1.0,Aiechhu,90.464808,26.945874,1963.905886,0.002897,2.940004e+09,0.000065,92.0,4160.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000763
1,3.0,Mangdechhu,90.649025,27.506149,7436.605311,0.004425,4.771824e+09,0.000051,109.0,7065.0,...,0.0,0.0,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.000000
2,4.0,Jaldhakha,88.987850,27.061616,1038.942931,0.002133,2.869328e+09,0.000066,210.0,4577.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000208
3,5.0,Amochhu,89.120110,27.350269,3929.619367,0.004123,2.904399e+09,0.000066,164.0,6689.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000106
4,6.0,Wangchhu,89.482723,27.353115,4608.336971,0.004038,3.551970e+09,0.000059,117.0,6689.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000014
5,7.0,Drangmechhu,91.395029,27.842435,21098.266822,0.009023,3.256464e+09,0.000062,90.0,7127.0,...,0.0,0.0,0.0,0.000022,0.0,0.0,0.0,0.0,0.0,0.000000
6,8.0,Punatsangchhu,89.939466,27.562875,9765.747380,0.005941,3.476406e+09,0.000060,94.0,7087.0,...,0.0,0.0,0.0,0.000007,0.0,0.0,0.0,0.0,0.0,0.000043
7,9.0,Nyera_Amari,91.652809,26.987295,2262.233550,0.003570,2.230784e+09,0.000075,99.0,4462.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
8,10.0,Jomori,91.959758,27.123484,750.319391,0.001549,3.931682e+09,0.000057,217.0,4477.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
9,NaN,Merak_Sakteng,92.033958,27.329946,138.810677,0.000645,4.188638e+09,0.000055,2690.0,4483.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
